# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guided, step-by-step template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID: {metadata.id}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Below, we print out the list of record set IDs, and for each record set, all the field IDs. This helps reference fields and record sets unambiguously for all downstream analyses.

In [ ]:
# Get all record sets and their IDs

record_set_ids = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print('Available record sets in this dataset:')
    for rs in metadata.recordSet:
        print(f"- Record set @id: {rs.id}, Name: {getattr(rs, 'name', '(no name)')}")
        record_set_ids.append(rs.id)
else:
    # If not present, we need to check what record sets the dataset actually exposes at runtime
    print('No record sets listed in metadata; discovering available record sets dynamically:')
    record_sets = dataset.record_sets
    for rs in record_sets:
        print(f"- Record set @id: {rs.id}, Name: {getattr(rs, 'name', '(no name)')}")
        record_set_ids.append(rs.id)

print("\nListing fields for each record set:")
field_ids_by_recordset = {}
for record_set_id in record_set_ids:
    print(f"\nRecord set @id: {record_set_id}")
    # Get a single record to infer fields
    try:
        records = dataset.records(record_set=record_set_id)
        record = next(records)
        print("Fields (column names / field @id):", list(record.keys()))
        field_ids_by_recordset[record_set_id] = list(record.keys())
    except Exception as e:
        print("Could not read records from record set.")
        field_ids_by_recordset[record_set_id] = []

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We recommend referencing the record set and field `@id`s from the overview above.

Below, we load all available record sets to separate Pandas DataFrames, using `@id`s for mapping.

In [ ]:
# Extract data from all discovered record sets
dataframes = {}
for record_set_id in record_set_ids:
    try:
        print(f"Loading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Number of records: {len(df)}  - Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Example: preview the first (or only) record set DataFrame
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nPreview of record set @id: {first_rs}")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data using only `@id` references. Replace field IDs below as applicable from the Data Overview.

In [ ]:
# For demonstration, we work on the first record set found:
selected_record_set_id = list(dataframes.keys())[0] if dataframes else None

df = dataframes[selected_record_set_id].copy()
print(f"Record set @id: {selected_record_set_id} has columns: {df.columns.tolist()}")

# Attempt to pick a numeric field (guess by column dtypes)
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field @id: {numeric_field_id}")
else:
    print("No numeric fields found to demonstrate EDA.")
    numeric_field_id = None

if numeric_field_id:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try finding a group-by candidate (categorical or string)
    possible_group_fields = df.select_dtypes(include=['object']).columns.tolist()
    group_field_id = possible_group_fields[0] if possible_group_fields else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("Skipping numeric field operations due to lack of numeric columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using only `@id` column/field references from previous steps.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field, if any, as histogram
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
else:
    print("No numeric field available for visualization.")

# Example scatter plot if at least two numeric fields
if len(numeric_fields) >= 2:
    plt.figure(figsize=(6,6))
    sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
    plt.xlabel(numeric_fields[0])
    plt.ylabel(numeric_fields[1])
    plt.title(f'Scatter plot: {numeric_fields[0]} vs {numeric_fields[1]}' )
    plt.show()
else:
    print("Not enough numeric fields for scatter plot.")

## 6. Conclusion
In this notebook, you have:
- Loaded a FAIR² Croissant-formatted dataset using `mlcroissant`
- Explored its metadata, available record sets, and field `@id`s
- Loaded tabular data as Pandas DataFrames using `@id` references
- Applied exploratory filtering, normalization, grouping, and created simple visualizations

Refer to the dataset and fields by `@id` for full reproducibility and semantic clarity. You may extend the workflow for downstream analysis, statistical tests, or more advanced ML pipelines using these standardized references.